In [ ]:
# For Question 1, you may find the following packages useful:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.model_selection import train_test_split

# 1.a) Import Housing.csv as a pandas DataFrame.
# Separate the explanatory features X from the target y = SalePrice.
#
# Guidance:
# - Refer to data_description.txt to determine how variables should be interpreted.
# - The first column of the csv file is an index and should not be used as an explanatory variable.
# - Keep the original variable names so that later coefficient tables are easy to interpret.


# Load the raw dataset; blank CSV fields become missing values, while literal labels are preserved.
df = pd.read_csv("Housing.csv", index_col=0, keep_default_na=False, na_values=[""])

# Separate target (y) and explanatory features (X)
y = df["SalePrice"]
X = df.drop(columns=["SalePrice"])

# Verify the shapes of X and y
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


# 1.b) Graphically determine whether SalePrice is approximately Gaussian.
# If it is not, suggest and apply a suitable transformation.
# Explain why considering such a transformation can be important.
#
# Guidance:
# - Use suitable graphical diagnostics (for example, a histogram and/or Q-Q plot).
# - Keep track of the transformation you choose because later questions require metrics
#   both on the transformed scale and, after inverse transformation, on the original
#   SalePrice scale.

# Graphical diagnostics for raw SalePrice
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Original SalePrice Histogram & Q-Q Plot
axes[0, 0].hist(y, bins=30, color='skyblue', edgecolor='black')
axes[0, 0].set_title("Histogram of SalePrice (Original)")
axes[0, 0].set_xlabel("SalePrice")
axes[0, 0].set_ylabel("Frequency")

stats.probplot(y, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title("Q-Q Plot of SalePrice (Original)")

# Apply log transformation
y_log = np.log(y)

# Transformed log(SalePrice) Histogram & Q-Q Plot
axes[1, 0].hist(y_log, bins=30, color='salmon', edgecolor='black')
axes[1, 0].set_title("Histogram of log(SalePrice)")
axes[1, 0].set_xlabel("log(SalePrice)")
axes[1, 0].set_ylabel("Frequency")

stats.probplot(y_log, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("Q-Q Plot of log(SalePrice)")

plt.tight_layout()
plt.show()

# Print skewness before and after
print(f"Original SalePrice Skewness: {y.skew():.3f}")
print(f"Log-transformed SalePrice Skewness: {y_log.skew():.3f}")

In [ ]:
# 1.c) Split the data into training and test sets.
# Randomly assign 70% of the observations to the training set and 30% to the test set.
#
# IMPORTANT:
# - Save an UNPROCESSED copy of X_train and X_test before doing any imputation,
#   standardization, or one-hot encoding.
# - You will return to these raw splits in Question 3.b) to build a leakage-safe
#   cross-validation pipeline.
#
# Suggested naming:
# X_train_raw, X_test_raw, y_train, y_test

# train_test_split shuffles by default and applies the same split to each input,
# keeping features and both target versions paired; test_size=0.30 reserves 30% for testing.
# random_state makes the split reproducible.
(
    X_train_raw,
    X_test_raw,

    y_train_log,
    y_test_log,
    
    y_train_original,
    y_test_original,
) = train_test_split(
    X,
    y_log,
    y,
    test_size=0.30,
    random_state=42,
)

# Use the transformed target for regression and retain original prices for evaluation.
y_train = y_train_log
y_test = y_test_log





# 1.d) Replace missing values in X using the training data statistics only:
#
# Setup needed before imputing: preserve the raw splits, then interpret documented
# NA/None labels before detecting numeric and categorical columns.
X_train = X_train_raw.copy()
X_test = X_test_raw.copy()

# NOTE: Some categorical variables admit ’NA’ (or ’None’) as a valid category, which should be
#   treated as an actual level and not as missing.
valid_na_categories = {
    "Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "PoolQC", "Fence", "MiscFeature",
}
valid_none_categories = {"MasVnrType"}
for split in (X_train, X_test):
    for column in split.columns:
        if column not in valid_na_categories:
            split[column] = split[column].replace("NA", np.nan)
        if column not in valid_none_categories:
            split[column] = split[column].replace("None", np.nan)
    for column in ("LotFrontage", "MasVnrArea", "GarageYrBlt"):
        split[column] = pd.to_numeric(split[column])

# NOTE: MSSubClass is categorical in the data description, although it is stored as a number.
categorical_columns = [
    column for column in X_train.columns
    if not pd.api.types.is_numeric_dtype(X_train[column])
]
categorical_columns.append("MSSubClass")
numeric_columns = [column for column in X_train.columns if column not in categorical_columns]

# – For numerical features, replace missing values with the mean of the column (computed
#   from the training set).
# Fit means on training data only, then apply those same means to train and test.
numeric_means = X_train[numeric_columns].mean()
X_train[numeric_columns] = X_train[numeric_columns].fillna(numeric_means)
X_test[numeric_columns] = X_test[numeric_columns].fillna(numeric_means)

# – For categorical features, replace missing values with the most frequent category in the
#   corresponding column (computed from the training set).
# Fit one mode per categorical column on training data, then use it for both splits:
categorical_modes = X_train[categorical_columns].mode(dropna=True).iloc[0]
X_train[categorical_columns] = X_train[categorical_columns].fillna(categorical_modes)
X_test[categorical_columns] = X_test[categorical_columns].fillna(categorical_modes)



# Next, standardize all numerical features by applying a z-score transform using the training
# mean and standard deviation, and apply the same transformation (using these training set
# statistics) to the test data. Finally, use one-hot encoding for all categorical features
# (e.g., with pd.get_dummies). After encoding, ensure that the test set has the same number of
# columns as the training set and the features appear in the correct order.
#
# Hint: If pd.get_dummies is applied separately to the training and test sets, they may
# produce different dummy-variable columns because some categorical levels occur in only one
# of the two sets. The regression model must receive the same feature columns at training and
# prediction time. You can use
# X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
# to make the encoded test columns match the training columns and their order. Dummy
# variables corresponding to training categories that are absent from the test set are then
# filled with zeros.

# StandardScaler uses only the training data to learn each feature's mean and population standard deviation.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# Apply z = (x - training_mean) / training_standard_deviation to training data.
X_train[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])
# Reuse the same training means and standard deviations for the test data.
X_test[numeric_columns] = scaler.transform(X_test[numeric_columns])

# One-hot encode each split, then align test columns to the training columns as in the hint.
X_train = pd.get_dummies(X_train, columns=categorical_columns, dtype=int)
X_test = pd.get_dummies(X_test, columns=categorical_columns, dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)




In [ ]:
# 2.a) Fit an OLS regression model using only the numerical features. Report the estimated coefficients and the model performance metrics (MSE and R²) on both the training and test sets. Report these metrics on both the transformed (log) scale and, after back-transforming, on the original SalePrice scale.

# Keep only the numerical predictors
X_train_num = X_train[numeric_columns]
X_test_num  = X_test[numeric_columns]

# Fit OLS with intercept
ols_num = LinearRegression(fit_intercept=True)
ols_num.fit(X_train_num, y_train)

# --- Coefficient table -------------------------------------------------
coefficients = pd.DataFrame({
    'Feature': ['Intercept'] + list(X_train_num.columns),
    'Coefficient': [ols_num.intercept_] + list(ols_num.coef_)
})
print("--- Estimated coefficients (OLS, numerical features only) ---")
display(coefficients)

# --- Predictions on the TRANSFORMED (log) scale ------------------------
y_train_pred_log = ols_num.predict(X_train_num)
y_test_pred_log  = ols_num.predict(X_test_num)

mse_train_log = mean_squared_error(y_train, y_train_pred_log)
mse_test_log  = mean_squared_error(y_test,  y_test_pred_log)
r2_train_log  = r2_score(y_train, y_train_pred_log)
r2_test_log   = r2_score(y_test,  y_test_pred_log)

# --- Back-transform to the ORIGINAL SalePrice scale --------------------
# NB: exp(y_hat) estimates the conditional MEDIAN under log-normal errors,
#     not the mean (see the markdown discussion).
y_train_orig      = np.exp(y_train)
y_test_orig       = np.exp(y_test)
y_train_pred_orig = np.exp(y_train_pred_log)
y_test_pred_orig  = np.exp(y_test_pred_log)

mse_train_orig = mean_squared_error(y_train_orig, y_train_pred_orig)
mse_test_orig  = mean_squared_error(y_test_orig,  y_test_pred_orig)
r2_train_orig  = r2_score(y_train_orig, y_train_pred_orig)
r2_test_orig   = r2_score(y_test_orig,  y_test_pred_orig)

results_table = pd.DataFrame({
    'Metric': ['MSE', 'R²'],
    'Training (log)':      [mse_train_log,  r2_train_log],
    'Testing (log)':       [mse_test_log,   r2_test_log],
    'Training (original)': [mse_train_orig, r2_train_orig],
    'Testing (original)':  [mse_test_orig,  r2_test_orig],
})
print("\n--- Model performance summary ---")
display(results_table)

#2.b (i) Normal equations via np.linalg.solve (as prescribed)
# ------------------------------------------------------------------
# Design matrix A = [1 | X_num] and target vector y (training only)
# ------------------------------------------------------------------
A    = np.column_stack([np.ones(X_train_num.shape[0]), X_train_num.values])
y_tr = y_train.values

m = A.shape[0]        # observations            -> 1022
p = A.shape[1]        # parameters d+1          -> 36
d = p - 1             # predictors              -> 35

# sklearn reference from 2.a as a flat vector [intercept, coef...]
sklearn_coefs = np.insert(ols_num.coef_, 0, ols_num.intercept_)

# ---- (i) Normal equations via np.linalg.solve (as prescribed) --------
# Kept to DEMONSTRATE the numerical failure under rank deficiency.
beta_solve = np.linalg.solve(A.T @ A, A.T @ y_tr)
print("--- (i) Normal equations with np.linalg.solve ---")
print(f"max |beta_solve - sklearn| = {np.max(np.abs(beta_solve - sklearn_coefs)):.4e}")
print(f"np.allclose vs sklearn?     {np.allclose(beta_solve, sklearn_coefs)}")

# ---- (i, stable) Minimum-norm solution via SVD (np.linalg.lstsq) -----
beta_svd, _, _, _ = np.linalg.lstsq(A, y_tr, rcond=None)

comparison_df = pd.DataFrame({
    'Feature': ['Intercept'] + list(X_train_num.columns),
    'NumPy (SVD / lstsq)': beta_svd,
    'scikit-learn': sklearn_coefs,
    'Absolute difference': np.abs(beta_svd - sklearn_coefs),
})
print("\n--- Coefficients: NumPy (SVD) vs scikit-learn ---")
display(comparison_df)
print(f"Max abs difference: {np.max(np.abs(beta_svd - sklearn_coefs)):.2e}")
print(f"Coincide within tolerance? {np.allclose(beta_svd, sklearn_coefs)}")

# ---- (ii) Standard errors --------------------------------------------
y_hat     = A @ beta_svd
residuals = y_tr - y_hat
rss       = residuals @ residuals

# Residual variance. We use the assignment's dof = m-(d+1). Under rank
# deficiency statsmodels uses m-rank(A) instead; this is the ONLY source
# of the tiny SE discrepancy in (vi).
dof        = m - (d + 1)          # 1022 - 36 = 986
sigma2_hat = rss / dof

# (A^T A)^+ computed stably as A^+ (A^+)^T  -- never form A^T A.
A_pinv   = np.linalg.pinv(A)
cov_unit = A_pinv @ A_pinv.T      # = (A^T A)^+  by the Penrose identity
var_beta = np.clip(sigma2_hat * np.diag(cov_unit), 0.0, None)
se_beta  = np.sqrt(var_beta)

se_table = pd.DataFrame({
    'Parameter': ['Intercept (j=0)'] +
                 [f'{c} (j={i+1})' for i, c in enumerate(X_train_num.columns)],
    'Estimate (beta_hat)': beta_svd,
    'Std. Error': se_beta,
})
print("--- (ii) Coefficients and standard errors (j=0 is the intercept) ---")
print(f"Intercept: beta_0 = {beta_svd[0]:.6f}, SE = {se_beta[0]:.6f}\n")
display(se_table)

# ---- (iii) In-sample MSE and R^2 vs 2.a ------------------------------
mse_in_manual = np.mean(residuals ** 2)
tss           = np.sum((y_tr - y_tr.mean()) ** 2)
r2_in_manual  = 1.0 - rss / tss

check = pd.DataFrame({
    'Metric (in-sample)': ['MSE', 'R²'],
    'Matrix algebra':     [mse_in_manual, r2_in_manual],
    'scikit-learn (2.a)': [mse_train_log, r2_train_log],
    'Abs difference':     [abs(mse_in_manual - mse_train_log),
                           abs(r2_in_manual - r2_train_log)],
})
print("\n--- (iii) Verification of in-sample metrics ---")
display(check)
print(f"MSE match: {np.isclose(mse_in_manual, mse_train_log)}")
print(f"R²  match: {np.isclose(r2_in_manual, r2_train_log)}")


# ---- (iv) Numerical structure of A -----------------------------------
rank_A = np.linalg.matrix_rank(A)
sing_A = np.linalg.svd(A, compute_uv=False)
cond_A = sing_A[0] / sing_A[-1]

print("--- (iv) Numerical structure of A ---")
print(f"Shape of A: {A.shape[0]} x {A.shape[1]}")
print(f"Rank of A: {rank_A}  (full column rank would be {A.shape[1]})")
print(f"Full column rank? {rank_A == A.shape[1]}")
print(f"Condition number kappa(A): {cond_A:.2e}")
print("\nFive smallest singular values of A:")
for k, s in enumerate(sing_A[-5:], start=len(sing_A) - 4):
    print(f"  sigma_{k:2d} = {s:.4e}")

# ---- (v) Ordinary (normal-equations) route vs pseudoinverse ----------
beta_normal = np.linalg.solve(A.T @ A, A.T @ y_tr)   # unstable baseline
beta_pinv   = A_pinv @ y_tr                          # minimum-norm, stable

rss_normal    = np.sum((y_tr - A @ beta_normal) ** 2)
rss_pinv      = np.sum((y_tr - A @ beta_pinv)   ** 2)
sigma2_normal = rss_normal / dof
sigma2_pinv   = rss_pinv   / dof

# Standard errors computed two ways, both "pseudoinverse-based":
#   (1) THROUGH A^T A  -> squares the condition number
#   (2) via pinv(A)    -> stable;  (A^T A)^+ = A^+ (A^+)^T
se_via_AtA   = np.sqrt(np.clip(sigma2_pinv * np.diag(np.linalg.pinv(A.T @ A)), 0, None))
se_via_pinvA = np.sqrt(np.clip(sigma2_pinv * np.diag(A_pinv @ A_pinv.T),       0, None))

print("--- (v) beta: normal equations vs pseudoinverse ---")
print(f"max |beta_normal - beta_pinv| = {np.max(np.abs(beta_normal - beta_pinv)):.6f}")
print("\n--- (v) residual variance ---")
print(f"|sigma2_normal - sigma2_pinv| = {abs(sigma2_normal - sigma2_pinv):.2e}")
print("\n--- (v) standard errors: through A^T A vs through pinv(A) ---")
print(f"max |se_via_AtA - se_via_pinvA| = {np.max(np.abs(se_via_AtA - se_via_pinvA)):.2e}")


# ---- (vi) Validation with statsmodels --------------------------------
sm_res = sm.OLS(y_tr, A).fit()      # triggers a rank-deficiency warning
print(sm_res.summary())

beta_sm = sm_res.params
se_sm   = sm_res.bse

cmp_sm = pd.DataFrame({
    'Feature': (['Intercept'] + list(X_train_num.columns))[:10],
    'beta (pinv)':        beta_pinv[:10],
    'beta (statsmodels)': beta_sm[:10],
    'SE (pinv)':          se_beta[:10],
    'SE (statsmodels)':   se_sm[:10],
})
print("\n--- (vi) pinv vs statsmodels (first 10) ---")
display(cmp_sm)
print(f"beta match (atol=1e-4): {np.allclose(beta_pinv, beta_sm, atol=1e-4)}")
print(f"SE   match (atol=1e-4): {np.allclose(se_beta,  se_sm,  atol=1e-4)}")
print(f"\nstatsmodels residual dof: {int(sm_res.df_resid)} "
      f"(= m - rank = {m} - {rank_A});  our dof = {dof} (= m - (d+1))")